# Agentic Network RCA – Demo Notebook

This notebook demonstrates how to call the **agentic-network-rca** API
and how to run the pipeline locally (without the server).

> ⚠️ You need a valid `OPENAI_API_KEY` set in your environment or `.env` file.

## 1. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # project root

from dotenv import load_dotenv
load_dotenv("../.env")  # loads OPENAI_API_KEY etc.

print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))

## 2. Telemetry Ingestion

In [ ]:
from pipelines.telemetry_ingestion import ingest_logs

chunks = ingest_logs()
print(f"Loaded {len(chunks)} log chunks.")
print("\nSample chunk:")
print(chunks[0])

## 3. RAG Retrieval

In [ ]:
from rag.retrieval import retrieve_context

query = "router R17 packet loss OSPF"
results = retrieve_context(query, k=3)
print(f"Top 3 context chunks for: '{query}'\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] {r[:120]}...\n")

## 4. Full Pipeline (Offline)

In [ ]:
from agents.rca_workflow import run_rca_pipeline
import json

log_input = "Router R17 experiencing packet loss and CPU usage high"

result = run_rca_pipeline(log_input)
print(json.dumps(result, indent=2))

## 5. Calling the API (server must be running)

In [ ]:
import httpx, json

API_BASE = "http://localhost:8000"

payload = {"logs": "Network latency spike detected on switch SW04"}

response = httpx.post(f"{API_BASE}/analyze_network", json=payload, timeout=60)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 6. API Health & Metrics

In [ ]:
health = httpx.get(f"{API_BASE}/health")
print("Health:", health.json())

metrics = httpx.get(f"{API_BASE}/metrics")
# Show just the RCA-related lines
for line in metrics.text.splitlines():
    if "rca_" in line and not line.startswith("#"):
        print(line)